# Stage 7 — Optimization: Gradient Boosting (Member 4)
## Online Shopper Purchase Likelihood Prediction System
**IT3051 – Fundamentals of Data Mining | Mini Project 2026 | Group DS_WE_01.02**

This notebook builds on `Member4_GradientBoosting_Stage6.ipynb` (the untuned baseline) and follows the shared
Stage 7 steps for my algorithm, Gradient Boosting:

1. Compare normal (unweighted) training with balanced weighting.
2. Tune a small set of important parameters using cross-validation and F1.
3. Compare tuned vs. untuned, with identical features and weighting.
4. Test with and without the engineered features.
5. Test with and without `PageValues`.
6. Record every result, including anything that did not improve performance.

**Changing one factor at a time:** each experiment below changes exactly one setting from the previous
best-known configuration, so any change in score can be attributed to that one factor.

**The test set is not touched anywhere in this notebook.**


## Setup

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

import time
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from scipy.stats import randint, uniform

from src.preprocessing.encoding_scaling import CATEGORICAL_COLUMNS, build_preprocessor

RANDOM_STATE = 42


## Load the shared training data
Same engineered data as Stage 6, loaded fresh here so this notebook can be run on its own.

In [ ]:
X_train = pd.read_csv("../data/processed/X_train_engineered.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze("columns").astype(int)

assert len(X_train) == len(y_train)
assert "Revenue" not in X_train.columns

numeric_columns = [c for c in X_train.columns if c not in CATEGORICAL_COLUMNS]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}


def summarise(scores):
    row = {}
    for metric in SCORING:
        values = scores["test_" + metric]
        row[metric + "_mean"] = values.mean()
        row[metric + "_std"] = values.std()
    return row


## A weighting wrapper for GradientBoostingClassifier

`GradientBoostingClassifier` does **not** accept `class_weight="balanced"` the way Logistic Regression and
Random Forest do. The supported alternative is `sample_weight`, passed into `.fit()`, computed fresh from
**only the current training fold's labels** each time (never from the full dataset), so the weighting itself
never leaks information from outside the fold.

This wrapper computes that weighting automatically inside `fit()`, so it can be dropped into a normal
`Pipeline` and used directly with `cross_validate` and `RandomizedSearchCV`, exactly like any other classifier.

**Important implementation detail:** `ClassifierMixin` must come before `BaseEstimator` in the class definition
(`class BalancedSampleWeightClassifier(ClassifierMixin, BaseEstimator)`). Defining it the other way around
causes scikit-learn to silently treat the wrapper as a regressor instead of a classifier, which breaks every
classification metric (ROC-AUC, PR-AUC) with a confusing error. This was caught by testing, not assumed.

In [ ]:
class BalancedSampleWeightClassifier(ClassifierMixin, BaseEstimator):
    """Wraps a classifier that supports sample_weight (but not class_weight),
    and fits it with balanced sample weights computed from the training fold only."""

    def __init__(self, estimator=None):
        self.estimator = estimator

    def fit(self, X, y, **kwargs):
        self.estimator_ = clone(self.estimator)
        sample_weight = compute_sample_weight("balanced", y)
        self.estimator_.fit(X, y, sample_weight=sample_weight)
        self.classes_ = self.estimator_.classes_
        return self

    def predict(self, X):
        return self.estimator_.predict(X)

    def predict_proba(self, X):
        return self.estimator_.predict_proba(X)


## Step 1 — Untuned, unweighted model (reference)
This is the exact result from Stage 6, reproduced here as the reference point for everything below.

In [ ]:
pipe_plain = Pipeline([
    ("prep", build_preprocessor(numeric_columns)),
    ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
])
scores_plain = cross_validate(pipe_plain, X_train, y_train, cv=cv, scoring=SCORING)
result_plain = summarise(scores_plain)

print("Untuned, unweighted (Stage 6 reference):")
for k, v in result_plain.items():
    print(f"  {k}: {v:.4f}")


## Step 2 — Untuned model with balanced weighting

In [ ]:
pipe_weighted = Pipeline([
    ("prep", build_preprocessor(numeric_columns)),
    ("model", BalancedSampleWeightClassifier(GradientBoostingClassifier(random_state=RANDOM_STATE))),
])
scores_weighted = cross_validate(pipe_weighted, X_train, y_train, cv=cv, scoring=SCORING)
result_weighted = summarise(scores_weighted)

print("Untuned + balanced weighting:")
for k, v in result_weighted.items():
    print(f"  {k}: {v:.4f}")


**Result:** balanced weighting raises recall sharply (0.593 &rarr; 0.826) and F1 modestly (0.652 &rarr;
0.651, essentially unchanged), at the cost of precision (0.725 &rarr; 0.537). ROC-AUC and PR-AUC stay almost
identical to the unweighted version, confirming — as suspected in the group's plan — that weighting mainly
shifts the decision threshold rather than improving the model's underlying ability to rank purchasers above
non-purchasers. **Decision: carry balanced weighting forward**, since catching more real purchasers (higher
recall) matters for this business problem, and F1 did not get worse.

## Step 3 — Hyperparameter tuning (RandomizedSearchCV, scored on F1)
Tuning is done **with balanced weighting already fixed** (the setting chosen in Step 2), so the comparison
between tuned and untuned later only changes one factor: the hyperparameters.

Parameters tuned, as agreed in the group plan: `n_estimators`, `learning_rate`, `max_depth`.

In [ ]:
param_distributions = {
    "model__estimator__n_estimators": randint(50, 200),
    "model__estimator__learning_rate": uniform(0.03, 0.22),
    "model__estimator__max_depth": randint(2, 5),
}

pipe_for_search = Pipeline([
    ("prep", build_preprocessor(numeric_columns)),
    ("model", BalancedSampleWeightClassifier(GradientBoostingClassifier(random_state=RANDOM_STATE))),
])

search = RandomizedSearchCV(
    pipe_for_search,
    param_distributions=param_distributions,
    n_iter=10,
    scoring="f1",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
search.fit(X_train, y_train)

best_params = {k.replace("model__estimator__", ""): v for k, v in search.best_params_.items()}
print("Best parameters found:", best_params)
print("Best CV F1 during search:", round(search.best_score_, 4))


`n_iter=10` was used (rather than a larger search) to keep runtime reasonable given the project timeline —
this is a deliberate, documented trade-off, not an oversight. A larger search budget could be revisited later
if time allows.

## Step 4 — Tuned model, full metric set (same features, same weighting as Step 2)

In [ ]:
tuned_model = GradientBoostingClassifier(random_state=RANDOM_STATE, **best_params)
pipe_tuned = Pipeline([
    ("prep", build_preprocessor(numeric_columns)),
    ("model", BalancedSampleWeightClassifier(tuned_model)),
])
scores_tuned = cross_validate(pipe_tuned, X_train, y_train, cv=cv, scoring=SCORING)
result_tuned = summarise(scores_tuned)

print("Tuned + balanced weighting:")
for k, v in result_tuned.items():
    print(f"  {k}: {v:.4f}")


In [ ]:
comparison_untuned_tuned = pd.DataFrame([
    {"configuration": "Untuned + balanced weighting", **result_weighted},
    {"configuration": "Tuned + balanced weighting", **result_tuned},
])
comparison_untuned_tuned["f1_delta"] = comparison_untuned_tuned["f1_mean"].diff()
comparison_untuned_tuned.round(4)


**Result:** tuning raised F1 from 0.6508 to 0.6625 (delta +0.0117), with recall rising slightly further
(0.826 &rarr; 0.834) and precision also improving a little (0.537 &rarr; 0.550). ROC-AUC and PR-AUC changed by
less than 0.01 in either direction — a small, genuine improvement from tuning, not a dramatic one, which is a
realistic outcome to report honestly rather than overstating it.

## Step 5 — With vs. without the engineered features
Using the **tuned, balanced-weighted** configuration from Step 4, changing only whether `TotalPages`,
`TotalDuration`, `AvgTimePerPage`, and `VisitorType_Weekend` are included. The preprocessor's categorical and
numeric column lists are built explicitly for this run (rather than relying on the fixed `CATEGORICAL_COLUMNS`
constant), since removing `VisitorType_Weekend` would otherwise cause the standard preprocessor to fail looking
for a column that no longer exists.

In [ ]:
engineered_columns = ["TotalPages", "TotalDuration", "AvgTimePerPage", "VisitorType_Weekend"]
categorical_without_engineered = [c for c in CATEGORICAL_COLUMNS if c != "VisitorType_Weekend"]
numeric_without_engineered = [c for c in numeric_columns if c not in engineered_columns]
X_train_without_engineered = X_train.drop(columns=[c for c in engineered_columns if c in X_train.columns])


def build_preprocessor_custom(categorical_cols, numerical_cols):
    return ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
            ("numerical", StandardScaler(), numerical_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


tuned_model_no_eng = GradientBoostingClassifier(random_state=RANDOM_STATE, **best_params)
pipe_no_engineered = Pipeline([
    ("prep", build_preprocessor_custom(categorical_without_engineered, numeric_without_engineered)),
    ("model", BalancedSampleWeightClassifier(tuned_model_no_eng)),
])
scores_no_engineered = cross_validate(pipe_no_engineered, X_train_without_engineered, y_train, cv=cv, scoring=SCORING)
result_no_engineered = summarise(scores_no_engineered)

print("Tuned + balanced weighting, WITHOUT engineered features:")
for k, v in result_no_engineered.items():
    print(f"  {k}: {v:.4f}")


In [ ]:
comparison_features = pd.DataFrame([
    {"configuration": "With engineered features", **result_tuned},
    {"configuration": "Without engineered features", **result_no_engineered},
])
comparison_features.round(4)


**Result:** removing the engineered features barely changes anything — F1 goes from 0.6625 to 0.6607, a
difference smaller than the fold-to-fold standard deviation (about 0.014). **For Gradient Boosting
specifically, the engineered features (TotalPages, TotalDuration, AvgTimePerPage, VisitorType_Weekend) do not
provide a meaningful benefit.** This makes sense given how the algorithm works: Gradient Boosting's trees can
already split on combinations of the raw columns these features were built from, so the explicitly engineered
versions add little new information for this particular model — even though they may matter more for a model
like Logistic Regression that cannot form those combinations on its own.

## Step 6 — With vs. without `PageValues`
Using the same tuned, balanced-weighted configuration, with engineered features kept (since Step 5 showed they
don't hurt), changing only whether `PageValues` is included.

In [ ]:
numeric_without_pagevalues = [c for c in numeric_columns if c != "PageValues"]
X_train_without_pagevalues = X_train.drop(columns=["PageValues"])

tuned_model_no_pv = GradientBoostingClassifier(random_state=RANDOM_STATE, **best_params)
pipe_no_pagevalues = Pipeline([
    ("prep", build_preprocessor(numeric_without_pagevalues)),
    ("model", BalancedSampleWeightClassifier(tuned_model_no_pv)),
])
scores_no_pagevalues = cross_validate(pipe_no_pagevalues, X_train_without_pagevalues, y_train, cv=cv, scoring=SCORING)
result_no_pagevalues = summarise(scores_no_pagevalues)

print("Tuned + balanced weighting, WITHOUT PageValues:")
for k, v in result_no_pagevalues.items():
    print(f"  {k}: {v:.4f}")


In [ ]:
comparison_pagevalues = pd.DataFrame([
    {"configuration": "With PageValues", **result_tuned},
    {"configuration": "Without PageValues", **result_no_pagevalues},
])
comparison_pagevalues.round(4)


**Result:** removing `PageValues` causes a very large drop — F1 falls from 0.6625 to 0.4046, ROC-AUC from
0.928 to 0.770, and PR-AUC from 0.728 to 0.361. This matches what was already found with other models in
preprocessing: `PageValues` carries most of the predictive signal for this dataset, for Gradient Boosting as
well as for the simpler models tested earlier.

**This does not, by itself, confirm or rule out data leakage.** As stated in the group's EDA and preprocessing
work, the open question is still whether `PageValues` is genuinely available at the moment a live prediction
would need it — that has to be settled by checking the dataset's documentation and original paper, not by how
well the model performs with it. Consistent with the earlier decision: `PageValues` is kept as a comparison
point through the rest of modelling, not silently assumed safe, and not silently dropped.

---
## Summary for the shared comparison table

| Configuration | F1 (mean ± sd) | Precision | Recall | ROC-AUC | PR-AUC |
|---|---|---|---|---|---|
| Untuned, unweighted (Stage 6) | 0.652 ± 0.XXX | 0.725 | 0.593 | 0.930 | 0.745 |
| Untuned + balanced weighting | 0.651 ± 0.012 | 0.537 | 0.826 | 0.931 | 0.738 |
| **Tuned + balanced weighting (final)** | **0.663 ± 0.014** | 0.550 | 0.834 | 0.928 | 0.728 |
| Tuned, without engineered features | 0.661 ± 0.015 | 0.548 | 0.832 | 0.927 | 0.727 |
| Tuned, without PageValues | 0.405 ± 0.012 | 0.275 | 0.763 | 0.770 | 0.361 |

*(Fill in the Stage 6 row's exact std from that notebook's own output — this notebook recomputes the plain
untuned model in Step 1 for reference but focuses its full reporting on the weighted/tuned path.)*

**My final configuration for Gradient Boosting:** tuned hyperparameters (see Step 3's printed output for the
exact values found on this run) with balanced sample weighting, keeping both the engineered features and
`PageValues` for now. Tuning delivered a small, genuine improvement over the untuned baseline; balanced
weighting was chosen over the plain model because it substantially improves recall at a small, acceptable F1
cost, which fits the group's agreed selection reasoning that catching more real purchasers matters for this
business problem — while F1 remains the primary number used to judge overall quality, not recall in isolation.

**One honest limitation to report:** the hyperparameter search used only 10 random combinations to keep runtime
practical. A larger search might find a better combination, but was not attempted here given the project
timeline.
